[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/pascal/pascal.ipynb)

# Pascal's Theorem

This example finds the conic through five points and checks Pascal's theorem on it. Five points in the plane, no three on a line, fix one conic through them. Add a sixth point on that conic and join the six in order into a hexagon: the three pairs of opposite sides meet in three points, and those three always lie on one line, however the hexagon is shaped, convex or crossed. Read backwards, the theorem builds the conic: the sixth points for which the three crossings line up are exactly the points of the conic through the other five.

A conic is a map from points to lines, `Plane <- Point`, sending each point to its polar line, and the conic is the set of points that lie on their own polar, `conic(point) & point == 0`. Two lines make a conic, their pair, and the conics through four points are the combinations of two such pairs; a fifth point fixes the combination. Pascal's condition, three crossings of joins joined into a number, is a form on the sixth point, which appears in two of the hexagon's sides, and it is the conic itself, up to scale.

In matrix notation a conic reads as a symmetric $3\times3$ matrix $C$ with $\mathbf p^{\mathsf T} C\,\mathbf p = 0$ on its points, the pencil through four points as the combination of two rank-two matrices, and the three crossings on one line as a vanishing $3\times3$ determinant.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga.algebras import PGA2D as ga
from numga import NumpyContext
from examples.util.animation import save_figure, show_animation
from examples.math.pascal import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Lines are vectors and points antivectors; a conic sends a point to its polar line.
Plane = ga.gatype.vector()
Point = ga.gatype.antivector()
Conic = ga.gatype((Plane, Point))                                              # Plane <- Point
Form = ga.gatype((Scalar, Point, Point))                                       # Scalar <- (Point, Point)
# The next vertex of each vertex of a hexagon.
NEXT = np.array([1, 2, 3, 4, 5, 0])

## 1. The conic through five points

Two lines make a conic that sends a point to the sum of each line weighted by the other's pairing with the point: its value at a point is the product of the two pairings, zero on either line. Through four points pass two pairs of lines, one pair of opposite sides of their quadrilateral and the pair of its diagonals, and every conic through the four is a combination of them. The fifth point fixes the combination: each pair weighted by the other's value there.

In [ ]:
# The ellipse the five points are taken from: its semi-axes along x and y, its centre, and their angles on it.
semi_axes = np.array([1.6, 1.0])
centre = np.array([0.3, -0.1])
five_angles = np.array([0.4, 1.44, 4.12, 4.77, 6.08])

In [ ]:
def pair(first: Plane, second: Plane) -> Conic:
    """Two lines as one conic: zero on the points of either line, its pairing the product of the two
    lines' pairings with the point."""
    return (first * (second & Point) + second * (first & Point)) / 2           # [...] Plane <- Point


def conic(points: Point) -> Conic:
    """The conic through five points on the last axis: the combination of the two line pairs through
    the first four that vanishes on the fifth."""
    first, second, third, fourth, fifth = (points[..., index] for index in range(5))
    sides = pair(first & second, third & fourth)                               # [...] Plane <- Point
    diagonals = pair(first & third, second & fourth)                           # [...] Plane <- Point
    return sides * (diagonals(fifth) & fifth) - diagonals * (sides(fifth) & fifth)   # [...] Plane <- Point


five = mv.point(centre + semi_axes * np.stack([np.cos(five_angles), np.sin(five_angles)], axis=-1))   # [5] Point
shape = conic(five)                                                            # [] Plane <- Point

In [ ]:
print(shape.coefficient_table())
print("the conic at the five points:", (shape(five) & five).to_array())

## 2. A sixth point, and the hexagon

A line through the first point meets the conic once more. Along the line the conic's value is a quadratic in the distance travelled with one root at the start, so the other root follows without a square root. Turning the line half a turn about the first point runs the sixth point once around the conic. With it, the six points make a hexagon, and its three pairs of opposite sides meet in three points.

In [ ]:
# The turning line's angles, half a turn in steps, and the step shown as a still.
n_frames = 120
angles = np.linspace(0.0, np.pi, n_frames, endpoint=False) + np.pi / (2 * n_frames)
still = 72
frame_duration = 60

In [ ]:
def second_crossing(shape: Conic, start: Point, heading: Point) -> Point:
    """Where the line from a point on a conic, towards a point at infinity, meets the conic again. Along
    `start + heading * t` the pairing is `2 * t * (shape(start) & heading) + t * t * (shape(heading) & heading)`,
    zero at the start and at the other root."""
    return start * (shape(heading) & heading) - heading * (2 * (shape(start) & heading))   # [...] Point


def headings(angles: np.ndarray) -> Point:
    """Points at infinity in the directions at the given angles."""
    return (mv.xy * (-angles / 2)).exp() >> mv.x.dual()


sixth = second_crossing(shape, five[0], headings(angles))                      # [frames] Point
sixth = sixth / (mv.w & sixth)                                                 # [frames] Point
hexagon = stack([five[index].broadcast_to(sixth.shape) for index in range(5)] + [sixth], axis=-1)   # [frames, 6] Point
# Each side the join of a corner and the next; opposite sides meet in the three crossings, and the line through
# them is Pascal's.
sides = hexagon & hexagon[..., NEXT]                                           # [frames, 6] Plane
crossing = sides[..., :3] ^ sides[..., 3:]                                     # [frames, 3] Point
pascal_line = crossing[..., 0] & crossing[..., 2]                              # [frames] Plane
save_figure(render.draw(shape, hexagon[still], sides[still], crossing[still], pascal_line[still]), "pascal_hexagon");

## 3. The crossings stay on one line

As the sixth point runs around, the hexagon changes shape, and the three crossings move, some far out when opposite sides are nearly parallel; the line through them, red, turns with them and never loses one. Their join, the regressive product of all three, is zero at every step.

In [ ]:
# The join of the three crossings, against their sizes, since nearly parallel sides meet far out.
join = crossing[..., 0] & crossing[..., 1] & crossing[..., 2]                   # [frames] Scalar
size = crossing[..., 0].norm() * crossing[..., 1].norm() * crossing[..., 2].norm()   # [frames] Scalar
show_animation(render.animate([(shape, hexagon[frame], sides[frame], crossing[frame], pascal_line[frame])
                               for frame in range(n_frames)]),
               "pascal_turn", frame_duration)

In [ ]:
print("largest join of the three crossings, against their sizes:", np.abs((join / size).to_array()).max())

## 4. Pascal's condition as the conic

Written as a condition on the sixth point, the three crossings joined into one number is a form on that point, which appears in two of the hexagon's sides, and so quadratic in it. It vanishes on every sixth point of the conic, and on nothing else: at any point at all it is the conic's value there, times one constant.

In [ ]:
# Points anywhere, on the conic or off it, and their seed.
n_probes = 8
probes = mv.point(np.random.default_rng(0).normal(size=(n_probes, 2)))        # [probes] Point

In [ ]:
def pascal(five: Point) -> Form:
    """Pascal's condition on a sixth point after five, a form on the sixth point, which appears in two of
    the hexagon's sides: the join of the three crossings of opposite sides, zero exactly when they lie
    on one line."""
    first, second, third, fourth, fifth = (five[..., index] for index in range(5))
    return (((first & second) ^ (fourth & fifth))
            & ((second & third) ^ (fifth & Point))
            & ((third & fourth) ^ (Point & first)))                            # [...] Scalar <- (Point, Point)


ratio = pascal(five)(probes, probes) / (shape(probes) & probes)                # [probes] Scalar

In [ ]:
print("Pascal's condition over the conic, at each probe:", ratio.to_array())

In [ ]:
# checks
# The conic passes through the five points and every sixth; the three crossings lie on one line; Pascal's
# condition is the conic, up to one constant.
np.testing.assert_allclose((shape(five) & five).to_array(), 0.0, atol=1e-11)
np.testing.assert_allclose((shape(hexagon) & hexagon).to_array(), 0.0, atol=1e-11)
np.testing.assert_allclose((join / size).to_array(), 0.0, atol=1e-10)
np.testing.assert_allclose(ratio.to_array(), ratio.to_array()[0], rtol=1e-11)